# Notebook 04 — Feature Engineering

## Objective

Membangun customer-level features dari transaction, customer profile,
dan session data yang telah dipersiapkan pada tahap sebelumnya.

## Business Context

Machine learning membutuhkan data dalam bentuk feature yang merepresentasikan
perilaku customer.

Pada notebook ini, data transaksi dan session akan diubah menjadi
customer-level analytical features.

## Main Feature Groups

1. RFM features
2. Purchase behavior
3. Customer tenure
4. Session engagement
5. Customer profile

## Methodology

Feature akan dibuat dengan melakukan aggregation berdasarkan `customer_id`.

### RFM

- Recency
- Frequency
- Monetary

### Purchase Behavior

- Total orders
- Total quantity
- Average order value

### Session Engagement

- Total sessions
- Average session duration
- Average pages viewed
- Total cart additions
- Conversion rate
- Bounce rate

## Important

Target `is_churned` tidak digunakan untuk menghitung behavioral features.

Kolom `segment` juga tidak digunakan sebagai predictive feature karena
telah diidentifikasi sebagai potential data leakage pada tahap EDA.

## Expected Output

Sebuah customer-level dataset yang dapat digunakan untuk:

- customer segmentation
- churn prediction
- business analysis

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Feature engineering environment initialized.")

Feature engineering environment initialized.


## 1. Load Analytical Data

### Tujuan

Memuat dataset yang diperlukan untuk feature engineering.

### Input

- `customer_analysis.csv`
- `completed_transactions.csv`
- `sessions.csv`

### Output

DataFrame yang akan digunakan untuk membangun customer-level features.

In [10]:
customer_analysis = pd.read_csv(
    PROCESSED_DIR / "customer_analysis.csv"
)

completed_transactions = pd.read_csv(
    PROCESSED_DIR / "completed_transactions.csv"
)

sessions = pd.read_csv(
    RAW_DIR / "sessions.csv"
)

customer_analysis["signup_date"] = pd.to_datetime(
    customer_analysis["signup_date"]
)

customer_analysis["first_purchase_date"] = pd.to_datetime(
    customer_analysis["first_purchase_date"]
)

customer_analysis["last_purchase_date"] = pd.to_datetime(
    customer_analysis["last_purchase_date"]
)

completed_transactions["transaction_date"] = pd.to_datetime(
    completed_transactions["transaction_date"]
)

sessions["session_date"] = pd.to_datetime(
    sessions["session_date"]
)

print("Customer analysis:", customer_analysis.shape)
print("Completed transactions:", completed_transactions.shape)
print("Sessions:", sessions.shape)

Customer analysis: (10000, 16)
Completed transactions: (68700, 13)
Sessions: (80000, 10)


## 2. RFM Features

### Tujuan

Membangun fitur RFM (Recency, Frequency, Monetary) untuk menggambarkan
nilai dan aktivitas customer berdasarkan riwayat transaksi.

### RFM Components

#### Recency

Mengukur berapa lama sejak customer melakukan transaksi terakhir.

Semakin kecil nilai Recency, semakin baru aktivitas pembelian customer.

#### Frequency

Mengukur jumlah transaksi yang dilakukan customer.

Semakin tinggi Frequency, semakin sering customer melakukan pembelian.

#### Monetary

Mengukur total revenue yang dihasilkan oleh customer.

Semakin tinggi Monetary, semakin besar kontribusi customer terhadap revenue.

### Input

`completed_transactions`

### Aggregation Level

Customer-level menggunakan `customer_id`.

### Expected Output

Feature:

- `recency_days`
- `frequency`
- `monetary`

### Important

RFM dihitung hanya menggunakan completed transactions.

In [11]:
reference_date = (
    completed_transactions["transaction_date"].max()
    + pd.Timedelta(days=1)
)

print("Latest transaction date:", completed_transactions["transaction_date"].max())
print("Reference date:", reference_date)

Latest transaction date: 2024-12-30 23:43:14
Reference date: 2024-12-31 23:43:14


### Build RFM Dataset

Customer-level RFM features dihitung dengan aggregation terhadap
completed transactions.

### Calculation

Recency:

`reference_date - last transaction date`

Frequency:

`jumlah completed transactions`

Monetary:

`total completed transaction revenue`

Setiap customer akan direpresentasikan sebagai satu baris.

In [12]:
rfm = (
    completed_transactions
    .groupby("customer_id")
    .agg(
        last_purchase_date=("transaction_date", "max"),
        frequency=("transaction_id", "nunique"),
        monetary=("total_amount", "sum"),
    )
    .reset_index()
)

rfm["recency_days"] = (
    reference_date - rfm["last_purchase_date"]
).dt.days

rfm = rfm[
    [
        "customer_id",
        "recency_days",
        "frequency",
        "monetary",
    ]
]

rfm.head()

,customer_id,recency_days,frequency,monetary
0,C00000,5,18,706.65
1,C00001,8,5,244.64
2,C00002,171,16,"1,104.66"
3,C00003,8,15,695.06
4,C00004,258,2,148.32


In [13]:
print("RFM shape:", rfm.shape)

print("\nMissing values:")
print(rfm.isna().sum())

print("\nRFM summary:")
display(rfm.describe())

RFM shape: (9730, 4)

Missing values:
customer_id     0
recency_days    0
frequency       0
monetary        0
dtype: int64

RFM summary:


,recency_days,frequency,monetary
count,"9,730.00","9,730.00","9,730.00"
mean,131.00,7.06,559.43
std,139.85,4.68,511.40
min,1.00,1.00,2.68
25%,33.00,4.00,200.00
50%,82.00,6.00,421.85
75%,177.00,10.00,761.21
max,730.00,30.00,"7,346.46"


## 3. Purchase Behavior Features

### Tujuan

Menambahkan fitur yang menggambarkan pola pembelian customer secara lebih
detail di luar RFM.

### Features

- `total_orders`
- `total_quantity`
- `average_order_value`

### Input

`completed_transactions`

### Aggregation Level

Customer-level menggunakan `customer_id`.

### Rationale

RFM menggunakan Frequency dan Monetary untuk menggambarkan aktivitas customer.

Pada bagian ini kita mempertahankan informasi tambahan mengenai jumlah
produk yang dibeli dan rata-rata nilai setiap transaksi.

Feature ini akan digunakan pada tahap customer analytics dan machine learning.

In [14]:
purchase_behavior = (
    completed_transactions
    .groupby("customer_id")
    .agg(
        total_orders=("transaction_id", "nunique"),
        total_quantity=("quantity", "sum"),
        average_order_value=("total_amount", "mean"),
    )
    .reset_index()
)

purchase_behavior.head()

,customer_id,total_orders,total_quantity,average_order_value
0,C00000,18,23,39.26
1,C00001,5,9,48.93
2,C00002,16,17,69.04
3,C00003,15,23,46.34
4,C00004,2,2,74.16


In [15]:
print("Purchase behavior shape:", purchase_behavior.shape)

print("\nMissing values:")
print(purchase_behavior.isna().sum())

print("\nPurchase behavior summary:")
display(purchase_behavior.describe())

Purchase behavior shape: (9730, 4)

Missing values:
customer_id            0
total_orders           0
total_quantity         0
average_order_value    0
dtype: int64

Purchase behavior summary:


,total_orders,total_quantity,average_order_value
count,"9,730.00","9,730.00","9,730.00"
mean,7.06,10.36,79.52
std,4.68,7.47,64.15
min,1.00,1.00,2.68
25%,4.00,5.00,45.30
50%,6.00,9.00,65.40
75%,10.00,14.00,93.66
max,30.00,65.00,"1,175.75"


In [16]:
frequency_check = rfm.merge(
    purchase_behavior[
        ["customer_id", "total_orders"]
    ],
    on="customer_id",
    how="inner",
)

frequency_check["difference"] = (
    frequency_check["frequency"]
    - frequency_check["total_orders"]
)

print(
    "Customers with frequency mismatch:",
    (frequency_check["difference"] != 0).sum()
)

frequency_check.head()

Customers with frequency mismatch: 0


,customer_id,recency_days,frequency,monetary,total_orders,difference
0,C00000,5,18,706.65,18,0
1,C00001,8,5,244.64,5,0
2,C00002,171,16,"1,104.66",16,0
3,C00003,8,15,695.06,15,0
4,C00004,258,2,148.32,2,0


## 4. Customer Tenure Features

### Tujuan

Mengukur lama customer telah terdaftar di platform.

Customer tenure dapat membantu membedakan customer baru dengan customer
yang telah lama berinteraksi dengan bisnis.

### Feature

- `tenure_days`

### Calculation

Tenure dihitung berdasarkan:

`reference_date - signup_date`

Reference date menggunakan tanggal yang sama dengan RFM sehingga seluruh
customer dibandingkan pada titik waktu yang konsisten.

### Input

`customer_analysis`

### Aggregation Level

Customer-level.

In [17]:
customer_tenure = (
    customer_analysis[
        ["customer_id", "signup_date"]
    ]
    .copy()
)

customer_tenure["tenure_days"] = (
    reference_date
    - customer_tenure["signup_date"]
).dt.days

customer_tenure = customer_tenure[
    ["customer_id", "tenure_days"]
]

customer_tenure.head()

,customer_id,tenure_days
0,C00000,1783
1,C00001,1187
2,C00002,919
3,C00003,1106
4,C00004,1202


In [18]:
print("Customer tenure shape:", customer_tenure.shape)

print("\nMissing values:")
print(customer_tenure.isna().sum())

print("\nTenure summary:")
display(customer_tenure["tenure_days"].describe())

print(
    "\nNegative tenure values:",
    (customer_tenure["tenure_days"] < 0).sum()
)

Customer tenure shape: (10000, 2)

Missing values:
customer_id    0
tenure_days    0
dtype: int64

Tenure summary:


count   10,000.00
mean       899.47
std        419.80
min          1.00
25%        567.75
50%        901.00
75%      1,229.00
max      1,826.00
Name: tenure_days, dtype: float64


Negative tenure values: 0


## 5. Session Engagement Features

### Tujuan

Mengubah data session-level menjadi customer-level features yang
merepresentasikan tingkat engagement customer.

### Features

- `total_sessions`
- `avg_session_duration`
- `avg_pages_viewed`
- `total_cart_additions`
- `conversion_rate`
- `bounce_rate`

### Input

`sessions`

### Aggregation Level

Customer-level menggunakan `customer_id`.

### Calculation

#### Total Sessions

Jumlah session yang dilakukan customer.

#### Average Session Duration

Rata-rata durasi session dalam detik.

#### Average Pages Viewed

Rata-rata jumlah halaman yang dilihat dalam session.

#### Total Cart Additions

Total aktivitas penambahan produk ke cart.

#### Conversion Rate

Proporsi session yang menghasilkan conversion.

#### Bounce Rate

Proporsi session yang menghasilkan bounce.

### Important

Session features digunakan untuk menggambarkan engagement behavior.

Features ini tidak menggunakan `is_churned` sebagai input.

In [19]:
session_features = (
    sessions
    .groupby("customer_id")
    .agg(
        total_sessions=("session_id", "nunique"),
        avg_session_duration=("duration_seconds", "mean"),
        avg_pages_viewed=("pages_viewed", "mean"),
        total_cart_additions=("cart_additions", "sum"),
        conversion_rate=("converted", "mean"),
        bounce_rate=("bounced", "mean"),
    )
    .reset_index()
)

session_features.head()

,customer_id,total_sessions,avg_session_duration,avg_pages_viewed,total_cart_additions,conversion_rate,bounce_rate
0,C00000,8,696.88,10.75,5,0.12,0.12
1,C00001,12,325.50,6.83,4,0.00,0.08
2,C00002,29,439.90,7.48,31,0.10,0.07
3,C00003,17,284.59,5.18,20,0.12,0.18
4,C00004,1,216.00,5.00,0,0.00,0.00


In [20]:
print("Session features shape:", session_features.shape)

print("\nMissing values:")
print(session_features.isna().sum())

print("\nSession feature summary:")
display(session_features.describe())

Session features shape: (9824, 7)

Missing values:
customer_id             0
total_sessions          0
avg_session_duration    0
avg_pages_viewed        0
total_cart_additions    0
conversion_rate         0
bounce_rate             0
dtype: int64

Session feature summary:


,total_sessions,avg_session_duration,avg_pages_viewed,total_cart_additions,conversion_rate,bounce_rate
count,"9,824.00","9,824.00","9,824.00","9,824.00","9,824.00","9,824.00"
mean,8.14,299.25,5.20,6.99,0.08,0.11
std,5.39,174.46,3.22,6.49,0.12,0.16
min,1.00,10.00,1.00,0.00,0.00,0.00
25%,4.00,168.00,2.71,2.00,0.00,0.00
50%,7.00,273.00,4.60,5.00,0.00,0.05
75%,11.00,402.76,7.00,10.00,0.14,0.17
max,33.00,"1,397.50",22.33,45.00,1.00,1.00


In [21]:
print(
    "Invalid conversion rates:",
    (
        (session_features["conversion_rate"] < 0)
        | (session_features["conversion_rate"] > 1)
    ).sum()
)

print(
    "Invalid bounce rates:",
    (
        (session_features["bounce_rate"] < 0)
        | (session_features["bounce_rate"] > 1)
    ).sum()
)

Invalid conversion rates: 0
Invalid bounce rates: 0


## 6. Final Customer Feature Dataset

### Tujuan

Menggabungkan seluruh feature yang telah dibuat menjadi satu
customer-level analytical dataset.

### Feature Groups

#### Customer Profile

- age
- gender
- country
- email_opt_in
- has_app

#### RFM

- recency_days
- frequency
- monetary

#### Purchase Behavior

- total_orders
- total_quantity
- average_order_value

#### Customer Tenure

- tenure_days

#### Session Engagement

- total_sessions
- avg_session_duration
- avg_pages_viewed
- total_cart_additions
- conversion_rate
- bounce_rate

### Target

`is_churned` dipertahankan sebagai target untuk tahap churn prediction,
tetapi tidak digunakan untuk menghitung behavioral features.

### Join Strategy

`customer_analysis` digunakan sebagai base dataset sehingga seluruh
10,000 customer tetap dipertahankan.

Customer yang tidak memiliki completed transaction atau session akan
memiliki feature aggregation yang kosong dan akan ditangani secara
eksplisit pada tahap berikutnya.

In [22]:
customer_features = customer_analysis[
    [
        "customer_id",
        "age",
        "gender",
        "country",
        "email_opt_in",
        "has_app",
        "is_churned",
    ]
].copy()

customer_features = customer_features.merge(
    rfm,
    on="customer_id",
    how="left",
)

customer_features = customer_features.merge(
    purchase_behavior,
    on="customer_id",
    how="left",
)

customer_features = customer_features.merge(
    customer_tenure,
    on="customer_id",
    how="left",
)

customer_features = customer_features.merge(
    session_features,
    on="customer_id",
    how="left",
)

print("Final customer feature shape:", customer_features.shape)

customer_features.head()

Final customer feature shape: (10000, 20)


,customer_id,age,gender,country,email_opt_in,has_app,is_churned,recency_days,frequency,monetary,total_orders,total_quantity,average_order_value,tenure_days,total_sessions,avg_session_duration,avg_pages_viewed,total_cart_additions,conversion_rate,bounce_rate
0,C00000,28,M,BR,0,0,0,5.00,18.00,706.65,18.00,23.00,39.26,1783,8.00,696.88,10.75,5.00,0.12,0.12
1,C00001,22,Prefer not to say,FR,1,0,0,8.00,5.00,244.64,5.00,9.00,48.93,1187,12.00,325.50,6.83,4.00,0.00,0.08
2,C00002,30,F,US,1,1,0,171.00,16.00,"1,104.66",16.00,17.00,69.04,919,29.00,439.90,7.48,31.00,0.10,0.07
3,C00003,48,F,US,1,0,1,8.00,15.00,695.06,15.00,23.00,46.34,1106,17.00,284.59,5.18,20.00,0.12,0.18
4,C00004,37,M,CA,0,1,0,258.00,2.00,148.32,2.00,2.00,74.16,1202,1.00,216.00,5.00,0.00,0.00,0.00


### Handling Customers Without Activity

Tidak semua customer memiliki completed transaction atau session.

Untuk aggregation features, nilai `NaN` memiliki arti bahwa tidak terdapat
aktivitas yang tercatat.

Untuk feature berbasis jumlah aktivitas, nilai tersebut dapat direpresentasikan
sebagai `0`.

Feature yang akan menggunakan `0`:

- frequency
- monetary
- total_orders
- total_quantity
- total_sessions
- total_cart_additions
- conversion_rate
- bounce_rate

`recency_days` tidak langsung diubah menjadi 0 karena nilai 0 memiliki arti
customer baru saja melakukan transaksi.

Customer tanpa transaksi akan ditangani secara khusus pada tahap modeling.

In [26]:
# Customers without completed transactions
no_purchase_mask = customer_features["recency_days"].isna()

print(
    "Customers without completed transactions:",
    no_purchase_mask.sum()
)

# Customers without sessions
no_session_mask = customer_features["avg_session_duration"].isna()

print(
    "Customers without sessions:",
    no_session_mask.sum()
)

# Purchase-related features
customer_features["frequency"] = (
    customer_features["frequency"].fillna(0)
)

customer_features["monetary"] = (
    customer_features["monetary"].fillna(0)
)

customer_features["total_orders"] = (
    customer_features["total_orders"].fillna(0)
)

customer_features["total_quantity"] = (
    customer_features["total_quantity"].fillna(0)
)

customer_features["average_order_value"] = (
    customer_features["average_order_value"].fillna(0)
)

# Session-related features
customer_features["total_sessions"] = (
    customer_features["total_sessions"].fillna(0)
)

customer_features["avg_session_duration"] = (
    customer_features["avg_session_duration"].fillna(0)
)

customer_features["avg_pages_viewed"] = (
    customer_features["avg_pages_viewed"].fillna(0)
)

customer_features["total_cart_additions"] = (
    customer_features["total_cart_additions"].fillna(0)
)

customer_features["conversion_rate"] = (
    customer_features["conversion_rate"].fillna(0)
)

customer_features["bounce_rate"] = (
    customer_features["bounce_rate"].fillna(0)
)

Customers without completed transactions: 270
Customers without sessions: 176


### Purchase Activity Indicator

`recency_days` bernilai missing untuk customer yang belum memiliki completed transaction.

Missing value tersebut dipertahankan karena memiliki makna bisnis:
customer belum pernah melakukan pembelian selesai.

Untuk membedakan kondisi tersebut dari missing data biasa, dibuat fitur
`has_purchase`:

- `1` = customer memiliki minimal satu completed transaction
- `0` = customer belum memiliki completed transaction

`recency_days` tidak diimputasi pada dataset fitur utama. Penanganan khusus
akan dilakukan pada tahap modeling jika algoritma membutuhkan nilai numerik
tanpa missing value.

In [28]:
customer_features["has_purchase"] = (
    customer_features["recency_days"].notna().astype(int)
)

print("Purchase activity distribution:")
print(customer_features["has_purchase"].value_counts())

print("\nCustomers without purchase:")
print((customer_features["has_purchase"] == 0).sum())

Purchase activity distribution:
has_purchase
1    9730
0     270
Name: count, dtype: int64

Customers without purchase:
270


In [29]:
print("Final feature shape:", customer_features.shape)

print("\nMissing values:")
print(customer_features.isna().sum())

print("\nDuplicate customer IDs:")
print(customer_features["customer_id"].duplicated().sum())

print("\nPurchase activity:")
print(customer_features["has_purchase"].value_counts())

print("\nCustomers without purchase:")
print((customer_features["has_purchase"] == 0).sum())

print("\nCustomers without session:")
print(customer_features["total_sessions"].eq(0).sum())

print("\nChurn distribution:")
print(customer_features["is_churned"].value_counts())

print("\nRecency statistics for customers with purchase:")
print(
    customer_features.loc[
        customer_features["has_purchase"] == 1,
        "recency_days"
    ].describe()
)

Final feature shape: (10000, 21)

Missing values:
customer_id               0
age                       0
gender                    0
country                   0
email_opt_in              0
has_app                   0
is_churned                0
recency_days            270
frequency                 0
monetary                  0
total_orders              0
total_quantity            0
average_order_value       0
tenure_days               0
total_sessions            0
avg_session_duration      0
avg_pages_viewed          0
total_cart_additions      0
conversion_rate           0
bounce_rate               0
has_purchase              0
dtype: int64

Duplicate customer IDs:
0

Purchase activity:
has_purchase
1    9730
0     270
Name: count, dtype: int64

Customers without purchase:
270

Customers without session:
176

Churn distribution:
is_churned
0    8306
1    1694
Name: count, dtype: int64

Recency statistics for customers with purchase:
count   9,730.00
mean      131.00
std       139.85


## Save Customer Feature Dataset

Customer-level feature dataset telah selesai dibangun dan divalidasi.

Dataset ini akan menjadi input untuk tahap berikutnya, terutama:

- Customer Segmentation
- Churn Prediction
- Business Analysis

`recency_days` tetap memiliki missing value untuk customer yang belum pernah
melakukan completed transaction. Penanganan khusus terhadap kondisi tersebut
akan dilakukan pada tahap modeling jika diperlukan.

In [30]:
FEATURE_OUTPUT = PROCESSED_DIR / "customer_features.csv"

customer_features.to_csv(
    FEATURE_OUTPUT,
    index=False
)

print(f"Feature dataset saved to: {FEATURE_OUTPUT}")
print(f"Shape: {customer_features.shape}")

Feature dataset saved to: ..\data\processed\customer_features.csv
Shape: (10000, 21)


[TEST] Reload Dataset

In [31]:
test_features = pd.read_csv(FEATURE_OUTPUT)

print("Reloaded shape:", test_features.shape)

print("\nDuplicate customer IDs:")
print(test_features["customer_id"].duplicated().sum())

print("\nMissing values:")
print(test_features.isna().sum())

print("\nTarget distribution:")
print(test_features["is_churned"].value_counts())

print("\nPurchase activity:")
print(test_features["has_purchase"].value_counts())

Reloaded shape: (10000, 21)

Duplicate customer IDs:
0

Missing values:
customer_id               0
age                       0
gender                    0
country                   0
email_opt_in              0
has_app                   0
is_churned                0
recency_days            270
frequency                 0
monetary                  0
total_orders              0
total_quantity            0
average_order_value       0
tenure_days               0
total_sessions            0
avg_session_duration      0
avg_pages_viewed          0
total_cart_additions      0
conversion_rate           0
bounce_rate               0
has_purchase              0
dtype: int64

Target distribution:
is_churned
0    8306
1    1694
Name: count, dtype: int64

Purchase activity:
has_purchase
1    9730
0     270
Name: count, dtype: int64
